In [ ]:
!pip install -q transformers accelerate safetensors

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/hinemo

Mounted at /content/drive
checkpoints  DATASET_FOR_HINEMO  hidden_states	splits


In [ ]:
# ── CHANGE THIS PER RUN ─────────────────────────────────────────
MODEL_CONFIGS = [
    {
        "hf_id"   : "bert-base-multilingual-cased",
        "folder"  : "bert-base-multilingual-cased"
    },
    {
        "hf_id"   : "google/muril-base-cased",
        "folder"  : "muril-base-cased"
    },
    {
        "hf_id"   : "xlm-roberta-base",
        "folder"  : "xlm-roberta-base"
    },
]
# ────────────────────────────────────────────────────────────────

BASE_DIR          = "/content/drive/MyDrive/hinemo"
VAL_PATH          = f"{BASE_DIR}/splits/hinemo_val.csv"
BATCH_SIZE        = 32
MAX_LENGTH        = 128
RANDOM_SEED       = 524
SAMPLES_PER_CLASS = 1000   # 1000 × 4 = 4000 total

print(f"Models to extract: {[c['folder'] for c in MODEL_CONFIGS]}")
print(f"Samples per class: {SAMPLES_PER_CLASS}")
print(f"Total samples:     {SAMPLES_PER_CLASS * 4}")

Models to extract: ['bert-base-multilingual-cased', 'muril-base-cased', 'xlm-roberta-base']
Samples per class: 1000
Total samples:     4000


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")

Device: cuda
GPU: Tesla T4


In [ ]:
val_df = pd.read_csv(VAL_PATH)
print(f"Full val set: {len(val_df)} rows")
print(val_df["gpt_emotion"].value_counts())

# Stratified 4k subsample
sample_df = (
    val_df
    .groupby("gpt_emotion", group_keys=False)
    .apply(lambda x: x.sample(SAMPLES_PER_CLASS, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print(f"\nSubsample: {len(sample_df)} rows")
print(sample_df["gpt_emotion"].value_counts())
print(f"\nLambda distribution:")
print(sample_df["lambda"].describe())

# Check lambda bucket sizes — important for Phase 7b
low_lambda  = sample_df[sample_df["lambda"] < 0.33]
high_lambda = sample_df[sample_df["lambda"] > 0.66]
print(f"\nLow-λ samples (< 0.33):  {len(low_lambda)}")
print(f"High-λ samples (> 0.66): {len(high_lambda)}")
print(f"\nLow-λ emotion distribution:")
print(low_lambda["gpt_emotion"].value_counts())
print(f"\nHigh-λ emotion distribution:")
print(high_lambda["gpt_emotion"].value_counts())

Full val set: 6926 rows
gpt_emotion
joy        2658
anger      1855
disgust    1304
sadness    1109
Name: count, dtype: int64

Subsample: 4000 rows
gpt_emotion
anger      1000
disgust    1000
joy        1000
sadness    1000
Name: count, dtype: int64

Lambda distribution:
count    4000.000000
mean        0.424237
std         0.282876
min         0.000000
25%         0.166667
50%         0.483007
75%         0.642857
max         1.000000
Name: lambda, dtype: float64

Low-λ samples (< 0.33):  1347
High-λ samples (> 0.66): 946

Low-λ emotion distribution:
gpt_emotion
joy        546
sadness    435
disgust    198
anger      168
Name: count, dtype: int64

High-λ emotion distribution:
gpt_emotion
disgust    295
anger      271
sadness    230
joy        150
Name: count, dtype: int64


/tmp/ipykernel_983/1741325748.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(SAMPLES_PER_CLASS, random_state=RANDOM_SEED))


In [ ]:
def extract_hidden_states(model_path, sample_df, tokenizer, output_dir, label):
    print(f"\nLoading: {model_path}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        output_hidden_states=True,
        ignore_mismatched_sizes=True
    ).to(device).eval()

    all_hidden = []

    for i in tqdm(range(0, len(sample_df), BATCH_SIZE), desc=f"Extracting [{label}]"):
        batch_texts = sample_df["text"].iloc[i : i + BATCH_SIZE].tolist()

        encoded = tokenizer(
            batch_texts,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)

        cls_per_layer = torch.stack(
            [h[:, 0, :] for h in outputs.hidden_states[1:]],
            dim=1
        )

        all_hidden.append(cls_per_layer.cpu().numpy())

    hidden_array = np.concatenate(all_hidden, axis=0)

    save_path = f"{output_dir}/hidden_{label}_full.npy"
    np.save(save_path, hidden_array)
    print(f"Saved: {save_path}  |  Shape: {hidden_array.shape}")

    del model
    torch.cuda.empty_cache()

    return hidden_array

In [ ]:
for config in MODEL_CONFIGS:
    hf_id      = config["hf_id"]
    folder     = config["folder"]
    output_dir = f"{BASE_DIR}/hidden_states/{folder}"
    checkpoint = f"{BASE_DIR}/checkpoints/{folder}/best_model"

    os.makedirs(output_dir, exist_ok=True)

    # Save metadata once per model
    meta_path = f"{output_dir}/metadata_full.csv"
    if not os.path.exists(meta_path):
        sample_df[["id", "gpt_emotion", "lambda"]].to_csv(meta_path, index=False)
        print(f"Metadata saved: {meta_path}")

    tokenizer = AutoTokenizer.from_pretrained(hf_id)

    print(f"\n{'='*60}")
    print(f"MODEL: {folder}")
    print(f"{'='*60}")

    # Pretrained
    extract_hidden_states(hf_id, sample_df, tokenizer, output_dir, "pretrained")

    # Finetuned
    extract_hidden_states(checkpoint, sample_df, tokenizer, output_dir, "finetuned")

    print(f"\n✓ {folder} complete")

Metadata saved: /content/drive/MyDrive/hinemo/hidden_states/bert-base-multilingual-cased/metadata_full.csv


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Extracting [pretrained]: 100%|██████████| 125/125 [00:28<00:00,  4.34it/s]


Saved: /content/drive/MyDrive/hinemo/hidden_states/xlm-roberta-base/hidden_pretrained_full.npy  |  Shape: (4000, 12, 768)

Loading: /content/drive/MyDrive/hinemo/checkpoints/xlm-roberta-base/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Extracting [finetuned]: 100%|██████████| 125/125 [00:31<00:00,  3.98it/s]


Saved: /content/drive/MyDrive/hinemo/hidden_states/xlm-roberta-base/hidden_finetuned_full.npy  |  Shape: (4000, 12, 768)

✓ xlm-roberta-base complete


In [ ]:
print("VERIFICATION\n" + "="*50)

for config in MODEL_CONFIGS:
    folder     = config["folder"]
    output_dir = f"{BASE_DIR}/hidden_states/{folder}"

    for label in ["pretrained", "finetuned"]:
        path = f"{output_dir}/hidden_{label}_full.npy"
        if os.path.exists(path):
            arr = np.load(path)
            print(f"✓ {folder[:25]:<25} {label:<12} {arr.shape}")
        else:
            print(f"✗ MISSING: {path}")

    meta = pd.read_csv(f"{output_dir}/metadata_full.csv")
    print(f"  metadata_full: {len(meta)} rows, "
          f"emotions: {meta['gpt_emotion'].value_counts().to_dict()}\n")

VERIFICATION
✓ bert-base-multilingual-ca pretrained   (4000, 12, 768)
✓ bert-base-multilingual-ca finetuned    (4000, 12, 768)
  metadata_full: 4000 rows, emotions: {'anger': 1000, 'disgust': 1000, 'joy': 1000, 'sadness': 1000}

✓ muril-base-cased          pretrained   (4000, 12, 768)
✓ muril-base-cased          finetuned    (4000, 12, 768)
  metadata_full: 4000 rows, emotions: {'anger': 1000, 'disgust': 1000, 'joy': 1000, 'sadness': 1000}

✓ xlm-roberta-base          pretrained   (4000, 12, 768)
✓ xlm-roberta-base          finetuned    (4000, 12, 768)
  metadata_full: 4000 rows, emotions: {'anger': 1000, 'disgust': 1000, 'joy': 1000, 'sadness': 1000}

